In [2]:
"""
Generate a reproducible SQLite database for a fictional community sports service.

All data is synthetic and generated by this script. No external data is used.

Measurement types represented:
- Nominal: facility type, activity, member type, and booking status
- Ordinal: satisfaction rating (1 = Very dissatisfied to 5 = Very satisfied)
  and equipment condition (1 = Poor to 5 = Excellent)
- Interval: indoor facility temperature in degrees Celsius
- Ratio: attendee counts, duration, work hours, and costs

Run with Python 3. The script uses only the standard library.
"""

from datetime import date, datetime, time, timedelta
from pathlib import Path
import math
import random
import sqlite3


# A fixed seed makes the generated records reproducible.
SEED = 20261105
rng = random.Random(SEED)

# The database is saved in the folder from which this script is run.
DATABASE_PATH = Path.cwd() / "community_sports_centre.sqlite"
AS_OF_DATE = date(2026, 11, 5)

FACILITY_TYPES = ["Gym", "Court", "Pool", "Studio", "Hall"]

# Each facility type has a typical capacity and hourly rate.
FACILITY_DETAILS = {
    "Gym": (40, 25.00),
    "Court": (24, 18.00),
    "Pool": (32, 42.00),
    "Studio": (26, 22.00),
    "Hall": (60, 35.00),
}

# The facility type links each activity to a suitable booking space.
ACTIVITIES = [
    ("Strength training", "Gym", 1.50, 60),
    ("Circuit training", "Gym", 2.00, 60),
    ("Personal training", "Gym", 8.00, 60),
    ("Badminton", "Court", 2.00, 60),
    ("Basketball", "Court", 1.75, 90),
    ("Tennis", "Court", 2.50, 60),
    ("Lane swimming", "Pool", 3.50, 60),
    ("Swimming lessons", "Pool", 5.00, 45),
    ("Aqua fitness", "Pool", 4.50, 45),
    ("Yoga", "Studio", 3.00, 60),
    ("Pilates", "Studio", 3.50, 60),
    ("Dance class", "Studio", 3.00, 60),
    ("Indoor football", "Hall", 2.50, 90),
    ("Community meeting", "Hall", 0.00, 120),
]

ACTIVITIES_BY_FACILITY = {}
for activity_id, activity in enumerate(ACTIVITIES, start=1):
    ACTIVITIES_BY_FACILITY.setdefault(activity[1], []).append(activity_id)

EQUIPMENT_BY_FACILITY = {
    "Gym": ["Treadmill", "Exercise bike", "Weight bench", "Rowing machine"],
    "Court": ["Court net", "Scoreboard", "Sports flooring", "Goal frame"],
    "Pool": ["Pool pump", "Pool filter", "Lane divider", "Safety hoist"],
    "Studio": ["Sound system", "Exercise mat set", "Projector", "Lighting unit"],
    "Hall": ["Portable goal", "Sound system", "Folding table set", "Scoreboard"],
}

TECHNICIAN_SPECIALISMS = [
    "General maintenance",
    "Electrical systems",
    "Pool systems",
    "Fitness equipment",
]

MATERIALS = [
    ("Cleaning supplies", 8.50),
    ("Replacement cable", 14.00),
    ("Electrical fitting", 12.75),
    ("Pool filter parts", 28.00),
    ("Floor repair kit", 22.50),
    ("Fastener pack", 5.25),
    ("Safety signage", 9.00),
    ("Replacement belt", 18.75),
    ("Sealant", 7.50),
    ("Protective padding", 16.00),
    ("Sports netting", 24.00),
    ("Lighting component", 19.50),
]


def random_date(start: date, end: date) -> date:
    """Choose a date within the supplied range, including both endpoints."""
    days_between = (end - start).days
    return start + timedelta(days=rng.randint(0, days_between))


def create_schema(connection: sqlite3.Connection) -> None:
    """Create the tables, relationships, indexes, and integrity rules."""
    # Remove only tables owned by this generator, starting with dependent tables.
    tables_to_rebuild = [
        "booking_imports",
        "attendance",
        "maintenance_material_usage",
        "maintenance_crew_assignments",
        "maintenance_orders",
        "bookings",
        "equipment",
        "materials",
        "technicians",
        "members",
        "activities",
        "facilities",
        "centres",
    ]

    for table_name in tables_to_rebuild:
        connection.execute(f"DROP TABLE IF EXISTS {table_name}")

    connection.executescript(
        """
        CREATE TABLE centres (
            centre_id INTEGER PRIMARY KEY,
            centre_code TEXT NOT NULL UNIQUE,
            centre_name TEXT NOT NULL UNIQUE,
            district_name TEXT NOT NULL
        );

        CREATE TABLE facilities (
            facility_id INTEGER PRIMARY KEY,
            centre_id INTEGER NOT NULL,
            facility_code TEXT NOT NULL UNIQUE,
            facility_name TEXT NOT NULL,
            facility_type TEXT NOT NULL
                CHECK (facility_type IN ('Gym', 'Court', 'Pool', 'Studio', 'Hall')),
            capacity INTEGER NOT NULL CHECK (capacity > 0),
            hourly_rate_gbp REAL NOT NULL CHECK (hourly_rate_gbp >= 0),
            FOREIGN KEY (centre_id) REFERENCES centres(centre_id)
        );

        CREATE TABLE activities (
            activity_id INTEGER PRIMARY KEY,
            activity_name TEXT NOT NULL UNIQUE,
            facility_type TEXT NOT NULL
                CHECK (facility_type IN ('Gym', 'Court', 'Pool', 'Studio', 'Hall')),
            fee_per_person_gbp REAL NOT NULL CHECK (fee_per_person_gbp >= 0),
            typical_duration_minutes INTEGER NOT NULL
                CHECK (typical_duration_minutes BETWEEN 30 AND 180)
        );

        CREATE TABLE members (
            member_id INTEGER PRIMARY KEY,
            member_code TEXT NOT NULL UNIQUE,
            member_type TEXT NOT NULL
                CHECK (member_type IN ('Adult', 'Student', 'Family', 'Senior')),
            joined_on TEXT NOT NULL,
            membership_status TEXT NOT NULL
                CHECK (membership_status IN ('Active', 'Inactive'))
        );

        CREATE TABLE equipment (
            equipment_id INTEGER PRIMARY KEY,
            facility_id INTEGER NOT NULL,
            equipment_code TEXT NOT NULL UNIQUE,
            equipment_name TEXT NOT NULL,
            condition_grade INTEGER NOT NULL
                CHECK (condition_grade BETWEEN 1 AND 5),
            installed_year INTEGER NOT NULL
                CHECK (installed_year BETWEEN 2000 AND 2026),
            is_available INTEGER NOT NULL CHECK (is_available IN (0, 1)),
            FOREIGN KEY (facility_id) REFERENCES facilities(facility_id)
        );

        CREATE TABLE bookings (
            booking_id INTEGER PRIMARY KEY,
            member_id INTEGER NOT NULL,
            facility_id INTEGER NOT NULL,
            activity_id INTEGER NOT NULL,
            created_at TEXT NOT NULL,
            start_at TEXT NOT NULL,
            end_at TEXT NOT NULL,
            duration_minutes INTEGER NOT NULL
                CHECK (duration_minutes BETWEEN 30 AND 180),
            attendee_count INTEGER NOT NULL CHECK (attendee_count > 0),
            booking_status TEXT NOT NULL
                CHECK (
                    booking_status IN (
                        'Confirmed', 'Completed', 'Cancelled', 'No-show'
                    )
                ),
            booking_fee_gbp REAL NOT NULL CHECK (booking_fee_gbp >= 0),
            indoor_temperature_c REAL NOT NULL
                CHECK (indoor_temperature_c BETWEEN 5 AND 35),
            satisfaction_rating INTEGER
                CHECK (
                    satisfaction_rating IS NULL
                    OR satisfaction_rating BETWEEN 1 AND 5
                ),
            notes TEXT,
            FOREIGN KEY (member_id) REFERENCES members(member_id),
            FOREIGN KEY (facility_id) REFERENCES facilities(facility_id),
            FOREIGN KEY (activity_id) REFERENCES activities(activity_id),
            CHECK (end_at > start_at),
            CHECK (created_at <= start_at),
            CHECK (
                booking_status = 'Completed'
                OR satisfaction_rating IS NULL
            )
        );

        CREATE TABLE attendance (
            booking_id INTEGER PRIMARY KEY,
            attendance_status TEXT NOT NULL
                CHECK (attendance_status IN ('Attended', 'No-show')),
            actual_attendee_count INTEGER NOT NULL
                CHECK (actual_attendee_count >= 0),
            recorded_on TEXT NOT NULL,
            FOREIGN KEY (booking_id) REFERENCES bookings(booking_id),
            CHECK (
                (attendance_status = 'Attended' AND actual_attendee_count > 0)
                OR
                (attendance_status = 'No-show' AND actual_attendee_count = 0)
            )
        );

        CREATE TABLE technicians (
            technician_id INTEGER PRIMARY KEY,
            technician_code TEXT NOT NULL UNIQUE,
            specialism TEXT NOT NULL
                CHECK (
                    specialism IN (
                        'General maintenance',
                        'Electrical systems',
                        'Pool systems',
                        'Fitness equipment'
                    )
                )
        );

        CREATE TABLE maintenance_orders (
            maintenance_order_id INTEGER PRIMARY KEY,
            equipment_id INTEGER NOT NULL,
            priority_level INTEGER NOT NULL CHECK (priority_level BETWEEN 1 AND 4),
            order_status TEXT NOT NULL
                CHECK (order_status IN ('Open', 'In progress', 'Completed')),
            opened_on TEXT NOT NULL,
            completed_on TEXT,
            estimated_cost_gbp REAL NOT NULL CHECK (estimated_cost_gbp >= 0),
            work_summary TEXT NOT NULL,
            FOREIGN KEY (equipment_id) REFERENCES equipment(equipment_id),
            CHECK (completed_on IS NULL OR completed_on >= opened_on),
            CHECK (
                (order_status = 'Completed' AND completed_on IS NOT NULL)
                OR
                (order_status <> 'Completed' AND completed_on IS NULL)
            )
        );

        -- Each order can involve several technicians. The paired IDs prevent
        -- the same technician from being assigned twice to the same order.
        CREATE TABLE maintenance_crew_assignments (
            maintenance_order_id INTEGER NOT NULL,
            technician_id INTEGER NOT NULL,
            assigned_on TEXT NOT NULL,
            work_hours REAL NOT NULL CHECK (work_hours > 0),
            PRIMARY KEY (maintenance_order_id, technician_id),
            FOREIGN KEY (maintenance_order_id)
                REFERENCES maintenance_orders(maintenance_order_id),
            FOREIGN KEY (technician_id)
                REFERENCES technicians(technician_id)
        );

        CREATE TABLE materials (
            material_id INTEGER PRIMARY KEY,
            material_name TEXT NOT NULL UNIQUE,
            standard_unit_cost_gbp REAL NOT NULL
                CHECK (standard_unit_cost_gbp >= 0)
        );

        -- The composite key allows a material once per order and records its use.
        CREATE TABLE maintenance_material_usage (
            maintenance_order_id INTEGER NOT NULL,
            material_id INTEGER NOT NULL,
            quantity INTEGER NOT NULL CHECK (quantity > 0),
            unit_cost_gbp REAL NOT NULL CHECK (unit_cost_gbp >= 0),
            PRIMARY KEY (maintenance_order_id, material_id),
            FOREIGN KEY (maintenance_order_id)
                REFERENCES maintenance_orders(maintenance_order_id),
            FOREIGN KEY (material_id) REFERENCES materials(material_id)
        );

        -- Incoming records are kept separately so duplicate and missing
        -- source information can be examined without changing bookings.
        CREATE TABLE booking_imports (
            import_id INTEGER PRIMARY KEY,
            booking_id INTEGER NOT NULL,
            source_record_ref TEXT NOT NULL,
            received_on TEXT NOT NULL,
            source_booking_status TEXT NOT NULL
                CHECK (
                    source_booking_status IN (
                        'Confirmed', 'Completed', 'Cancelled', 'No-show'
                    )
                ),
            source_note TEXT,
            is_duplicate INTEGER NOT NULL CHECK (is_duplicate IN (0, 1)),
            FOREIGN KEY (booking_id) REFERENCES bookings(booking_id)
        );

        CREATE INDEX idx_bookings_facility_start
            ON bookings(facility_id, start_at);

        CREATE INDEX idx_maintenance_status
            ON maintenance_orders(order_status);

        CREATE INDEX idx_booking_imports_source_ref
            ON booking_imports(source_record_ref);
        """
    )


def generate_database() -> None:
    """Generate the synthetic records and save them to the SQLite file."""
    connection = sqlite3.connect(DATABASE_PATH)

    # SQLite enables foreign-key enforcement separately for each connection.
    connection.execute("PRAGMA foreign_keys = ON")

    try:
        create_schema(connection)

        # Eight fictional centres keep the records separate from real venues.
        districts = [
            "North District",
            "South District",
            "East District",
            "West District",
            "Central District",
            "Riverside District",
            "Hilltop District",
            "Meadow District",
        ]

        centre_rows = [
            (
                f"CSC-{centre_id:03d}",
                f"Westbridge Community Centre {centre_id:02d}",
                district,
            )
            for centre_id, district in enumerate(districts, start=1)
        ]

        connection.executemany(
            """
            INSERT INTO centres (centre_code, centre_name, district_name)
            VALUES (?, ?, ?)
            """,
            centre_rows,
        )

        # Each centre has one facility of each type.
        facility_rows = []
        facility_details = {}
        facility_id = 1

        for centre_id in range(1, len(centre_rows) + 1):
            for facility_type in FACILITY_TYPES:
                typical_capacity, typical_rate = FACILITY_DETAILS[facility_type]
                capacity = max(8, typical_capacity + rng.randint(-4, 6))
                hourly_rate = round(typical_rate * rng.uniform(0.90, 1.12), 2)

                facility_rows.append(
                    (
                        centre_id,
                        f"CSC-FAC-{facility_id:03d}",
                        f"Centre {centre_id:02d} {facility_type}",
                        facility_type,
                        capacity,
                        hourly_rate,
                    )
                )
                facility_details[facility_id] = {
                    "type": facility_type,
                    "capacity": capacity,
                    "hourly_rate": hourly_rate,
                }
                facility_id += 1

        connection.executemany(
            """
            INSERT INTO facilities (
                centre_id, facility_code, facility_name,
                facility_type, capacity, hourly_rate_gbp
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            facility_rows,
        )

        connection.executemany(
            """
            INSERT INTO activities (
                activity_name, facility_type,
                fee_per_person_gbp, typical_duration_minutes
            )
            VALUES (?, ?, ?, ?)
            """,
            ACTIVITIES,
        )

        # Members have synthetic codes and no real names or contact details.
        # Their join dates precede the generated booking history.
        member_rows = []
        active_member_ids = []

        for member_id in range(1, 501):
            joined_on = random_date(date(2019, 1, 1), date(2023, 12, 31))
            membership_status = (
                "Active" if rng.random() < 0.88 else "Inactive"
            )

            member_rows.append(
                (
                    f"CSC-MEM-{member_id:05d}",
                    rng.choice(["Adult", "Student", "Family", "Senior"]),
                    joined_on.isoformat(),
                    membership_status,
                )
            )

            if membership_status == "Active":
                active_member_ids.append(member_id)

        connection.executemany(
            """
            INSERT INTO members (
                member_code, member_type, joined_on, membership_status
            )
            VALUES (?, ?, ?, ?)
            """,
            member_rows,
        )

        # Equipment type is selected to match the facility where it is installed.
        equipment_rows = []
        equipment_details = {}

        for equipment_id in range(1, 201):
            assigned_facility_id = rng.randint(1, len(facility_rows))
            facility_type = facility_details[assigned_facility_id]["type"]
            equipment_name = rng.choice(EQUIPMENT_BY_FACILITY[facility_type])

            # Condition is ordinal: 1 is poor and 5 is excellent.
            condition_grade = rng.choices(
                [1, 2, 3, 4, 5],
                weights=[1, 2, 4, 6, 5],
                k=1,
            )[0]
            installed_year = rng.randint(2010, 2025)
            is_available = 0 if condition_grade == 1 else 1

            equipment_rows.append(
                (
                    assigned_facility_id,
                    f"CSC-EQP-{equipment_id:05d}",
                    equipment_name,
                    condition_grade,
                    installed_year,
                    is_available,
                )
            )
            equipment_details[equipment_id] = {
                "condition_grade": condition_grade,
                "equipment_name": equipment_name,
            }

        connection.executemany(
            """
            INSERT INTO equipment (
                facility_id, equipment_code, equipment_name,
                condition_grade, installed_year, is_available
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            equipment_rows,
        )

        technician_rows = [
            (
                f"CSC-TECH-{number:03d}",
                rng.choice(TECHNICIAN_SPECIALISMS),
            )
            for number in range(1, 13)
        ]

        connection.executemany(
            """
            INSERT INTO technicians (technician_code, specialism)
            VALUES (?, ?)
            """,
            technician_rows,
        )

        connection.executemany(
            """
            INSERT INTO materials (material_name, standard_unit_cost_gbp)
            VALUES (?, ?)
            """,
            MATERIALS,
        )

        # Bookings use an activity that matches the selected facility type.
        booking_rows = []
        booking_details = {}

        for booking_id in range(1, 1801):
            member_id = rng.choice(active_member_ids)
            selected_facility_id = rng.randint(1, len(facility_rows))
            facility = facility_details[selected_facility_id]

            compatible_activity_ids = ACTIVITIES_BY_FACILITY[facility["type"]]
            activity_id = rng.choice(compatible_activity_ids)
            activity = ACTIVITIES[activity_id - 1]

            booking_day = random_date(date(2024, 1, 1), date(2026, 12, 31))
            start_hour = rng.randint(6, 20)
            start_datetime = datetime.combine(
                booking_day,
                time(hour=start_hour),
            )

            duration_minutes = rng.choice([45, 60, 90, 120])
            end_datetime = start_datetime + timedelta(minutes=duration_minutes)
            created_datetime = start_datetime - timedelta(
                days=rng.randint(1, 90),
                hours=rng.randint(0, 12),
            )

            # Future sessions remain confirmed. Past sessions have an outcome.
            if booking_day > AS_OF_DATE:
                booking_status = "Confirmed"
            else:
                booking_status = rng.choices(
                    ["Completed", "Cancelled", "No-show"],
                    weights=[0.84, 0.09, 0.07],
                    k=1,
                )[0]

            attendee_count = rng.randint(
                1,
                min(facility["capacity"], 18),
            )

            duration_hours = duration_minutes / 60
            fee_per_person = activity[2]
            booking_fee = round(
                facility["hourly_rate"] * duration_hours
                + fee_per_person * attendee_count,
                2,
            )

            # Celsius is an interval measure. The seasonal shift is modest
            # because these are indoor facilities.
            seasonal_shift = 2.5 * math.sin(
                2 * math.pi * (booking_day.month - 4) / 12
            )
            indoor_temperature = round(
                19 + seasonal_shift + rng.uniform(-2.0, 2.0),
                1,
            )

            # Satisfaction is collected only for completed bookings.
            satisfaction_rating = None
            if booking_status == "Completed" and rng.random() < 0.78:
                satisfaction_rating = rng.randint(1, 5)

            # Optional notes are deliberately missing for some bookings.
            notes = None if rng.random() < 0.12 else rng.choice(
                [
                    "Standard booking.",
                    "Equipment set up before arrival.",
                    "Room checked after use.",
                    "Additional cleaning requested.",
                    "Accessibility support arranged.",
                ]
            )

            booking_rows.append(
                (
                    member_id,
                    selected_facility_id,
                    activity_id,
                    created_datetime.isoformat(timespec="minutes"),
                    start_datetime.isoformat(timespec="minutes"),
                    end_datetime.isoformat(timespec="minutes"),
                    duration_minutes,
                    attendee_count,
                    booking_status,
                    booking_fee,
                    indoor_temperature,
                    satisfaction_rating,
                    notes,
                )
            )

            booking_details[booking_id] = {
                "status": booking_status,
                "session_day": booking_day,
                "created_on": created_datetime.date(),
                "attendee_count": attendee_count,
                "notes": notes,
            }

        connection.executemany(
            """
            INSERT INTO bookings (
                member_id, facility_id, activity_id, created_at,
                start_at, end_at, duration_minutes, attendee_count,
                booking_status, booking_fee_gbp, indoor_temperature_c,
                satisfaction_rating, notes
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            booking_rows,
        )

        # Record attendance outcomes only for sessions that have already passed.
        # Cancelled bookings do not represent attendance records.
        attendance_rows = []

        for booking_id, booking in booking_details.items():
            if booking["session_day"] > AS_OF_DATE:
                continue

            if booking["status"] == "Completed":
                attendance_status = "Attended"
                actual_count = rng.randint(1, booking["attendee_count"])
            elif booking["status"] == "No-show":
                attendance_status = "No-show"
                actual_count = 0
            else:
                continue

            attendance_rows.append(
                (
                    booking_id,
                    attendance_status,
                    actual_count,
                    booking["session_day"].isoformat(),
                )
            )

        connection.executemany(
            """
            INSERT INTO attendance (
                booking_id, attendance_status,
                actual_attendee_count, recorded_on
            )
            VALUES (?, ?, ?, ?)
            """,
            attendance_rows,
        )

        # Poorer equipment condition makes a maintenance order more likely.
        equipment_needing_work = [
            equipment_id
            for equipment_id, details in equipment_details.items()
            if details["condition_grade"] <= 3
        ]

        maintenance_order_rows = []
        crew_assignment_rows = []
        material_usage_rows = []
        maintenance_order_id = 1

        for equipment_id in equipment_needing_work:
            opened_on = random_date(date(2024, 1, 1), AS_OF_DATE)
            days_available = (AS_OF_DATE - opened_on).days

            if days_available > 0 and rng.random() < 0.78:
                order_status = "Completed"
                completion_delay = rng.randint(1, min(35, days_available))
                completed_on = (
                    opened_on + timedelta(days=completion_delay)
                ).isoformat()
            else:
                order_status = rng.choice(["Open", "In progress"])
                completed_on = None

            condition_grade = equipment_details[equipment_id]["condition_grade"]
            priority_level = {
                1: 4,
                2: 3,
                3: 2,
                4: 1,
                5: 1,
            }[condition_grade]

            # A job may use multiple materials and technicians.
            chosen_material_ids = rng.sample(
                range(1, len(MATERIALS) + 1),
                k=rng.randint(1, 3),
            )
            chosen_technician_ids = rng.sample(
                range(1, 13),
                k=rng.randint(1, 2),
            )

            materials_cost = 0.0
            for material_id in chosen_material_ids:
                material_name, standard_cost = MATERIALS[material_id - 1]
                quantity = rng.randint(1, 5)
                unit_cost = round(standard_cost * rng.uniform(0.90, 1.12), 2)
                materials_cost += quantity * unit_cost

                material_usage_rows.append(
                    (
                        maintenance_order_id,
                        material_id,
                        quantity,
                        unit_cost,
                    )
                )

            labour_cost = 0.0
            for technician_id in chosen_technician_ids:
                work_hours = round(rng.uniform(1.0, 6.0), 1)
                labour_cost += work_hours * 26.00

                crew_assignment_rows.append(
                    (
                        maintenance_order_id,
                        technician_id,
                        opened_on.isoformat(),
                        work_hours,
                    )
                )

            estimated_cost = round(45.00 + materials_cost + labour_cost, 2)
            equipment_name = equipment_details[equipment_id]["equipment_name"]
            work_summary = f"Inspect and service {equipment_name.lower()}."

            maintenance_order_rows.append(
                (
                    equipment_id,
                    priority_level,
                    order_status,
                    opened_on.isoformat(),
                    completed_on,
                    estimated_cost,
                    work_summary,
                )
            )
            maintenance_order_id += 1

        connection.executemany(
            """
            INSERT INTO maintenance_orders (
                equipment_id, priority_level, order_status, opened_on,
                completed_on, estimated_cost_gbp, work_summary
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            maintenance_order_rows,
        )

        connection.executemany(
            """
            INSERT INTO maintenance_crew_assignments (
                maintenance_order_id, technician_id, assigned_on, work_hours
            )
            VALUES (?, ?, ?, ?)
            """,
            crew_assignment_rows,
        )

        connection.executemany(
            """
            INSERT INTO maintenance_material_usage (
                maintenance_order_id, material_id, quantity, unit_cost_gbp
            )
            VALUES (?, ?, ?, ?)
            """,
            material_usage_rows,
        )

        # Source notes are generated independently from booking notes so the
        # missing-value rate is controlled and easy to explain.
        import_rows = []

        for booking_id, booking in booking_details.items():
            if rng.random() < 0.10:
                source_note = None
            else:
                source_note = rng.choice(
                    [
                        "Imported from the booking system.",
                        "Booking details received successfully.",
                        "Source record contained standard booking information.",
                    ]
                )

            import_rows.append(
                (
                    booking_id,
                    f"SOURCE-BOOKING-{booking_id:05d}",
                    booking["created_on"].isoformat(),
                    booking["status"],
                    source_note,
                    0,
                )
            )

        # These extra rows deliberately repeat source references and values.
        duplicate_count = 36
        selected_originals = rng.sample(import_rows, k=duplicate_count)

        for original_row in selected_originals:
            duplicate_row = list(original_row)
            duplicate_row[5] = 1
            import_rows.append(tuple(duplicate_row))

        connection.executemany(
            """
            INSERT INTO booking_imports (
                booking_id, source_record_ref, received_on,
                source_booking_status, source_note, is_duplicate
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            import_rows,
        )

        connection.commit()

        # Print generated counts so they can be reported accurately.
        print(f"Database created: {DATABASE_PATH}")
        print(f"Random seed: {SEED}")
        print("\nGenerated row counts:")

        table_names = [
            "centres",
            "facilities",
            "activities",
            "members",
            "equipment",
            "bookings",
            "attendance",
            "technicians",
            "maintenance_orders",
            "maintenance_crew_assignments",
            "materials",
            "maintenance_material_usage",
            "booking_imports",
        ]

        for table_name in table_names:
            row_count = connection.execute(
                f"SELECT COUNT(*) FROM {table_name}"
            ).fetchone()[0]
            print(f"  {table_name}: {row_count:,}")

        missing_booking_notes = connection.execute(
            "SELECT COUNT(*) FROM bookings WHERE notes IS NULL"
        ).fetchone()[0]

        missing_completed_ratings = connection.execute(
            """
            SELECT COUNT(*)
            FROM bookings
            WHERE booking_status = 'Completed'
              AND satisfaction_rating IS NULL
            """
        ).fetchone()[0]

        missing_import_notes = connection.execute(
            "SELECT COUNT(*) FROM booking_imports WHERE source_note IS NULL"
        ).fetchone()[0]

        duplicate_import_rows = connection.execute(
            "SELECT COUNT(*) FROM booking_imports WHERE is_duplicate = 1"
        ).fetchone()[0]

        print("\nData-quality records:")
        print(f"  Bookings with missing optional notes: {missing_booking_notes}")
        print(
            "  Completed bookings without a satisfaction rating: "
            f"{missing_completed_ratings}"
        )
        print(f"  Imports with missing source notes: {missing_import_notes}")
        print(f"  Deliberate duplicate import rows: {duplicate_import_rows}")
        print("Foreign-key enforcement was enabled during generation.")

    finally:
        connection.close()


if __name__ == "__main__":
    generate_database()

Database created: /content/community_sports_centre.sqlite
Random seed: 20261105

Generated row counts:
  centres: 8
  facilities: 40
  activities: 14
  members: 500
  equipment: 200
  bookings: 1,800
  attendance: 1,548
  technicians: 12
  maintenance_orders: 80
  maintenance_crew_assignments: 125
  materials: 12
  maintenance_material_usage: 160
  booking_imports: 1,836

Data-quality records:
  Bookings with missing optional notes: 224
  Completed bookings without a satisfaction rating: 337
  Imports with missing source notes: 190
  Deliberate duplicate import rows: 36
Foreign-key enforcement was enabled during generation.
